# Mersenne Prime Cryptosystem

## SECTION 1.2 OUR CRYPTOSYSTEM

## Sem error correction: só para ver como funciona

In [ ]:
# SEM CÓDIGO DE CORREÇÃO DE ERROS

import secrets # Para geração de números aleatórios seguros

def low_hamming_weight_number(n, weight):
    """Gera um número de n bits com um Peso de Hamming específico."""
    bits = [0] * n
    indices = set()
    while len(indices) < weight:
        indices.add(secrets.randbelow(n)) #escolhe indices aleatorios para colocar o bit 1 - garantindo o peso de Hamming
    for i in indices:
        bits[i] = 1
    return int("".join(map(str, bits)), 2)

def mersenne_encrypt(m_bits, n, weight, p, R, T):
    """
    weight: número de bits 1 em A, B1, B2

    Simulação da criptografia:
    C1 = A*R + B1 (mod p)
    C2 = (A*T + B2) XOR E(m)
    """
    # Escolha de A, B1, B2 com baixo peso de Hamming
    A = low_hamming_weight_number(n, weight)
    B1 = low_hamming_weight_number(n, weight)
    B2 = low_hamming_weight_number(n, weight)
    
    # C1 = A * R + B1 mod p
    C1 = (A * R + B1) % p
    
    # C2 = (A * T + B2) XOR m #sem error correction 
    pad = (A * T + B2) % p
    C2 = pad ^ m_bits # XOR para combinar o pad com a mensagem
    
    return (C1, C2)

def mersenne_decrypt(C, F, p):
    """
    Desencriptar a mensagem:
    C2_star = C1 * F (mod p)
    m = D(C2 XOR C2_star)
    """
    C1, C2 = C
    C2_star = (C1 * F) % p
    
    # m_recuperado = C2 XOR C2_star
    m_bits = C2 ^ C2_star
    return m_bits

# --- Exemplo de Uso ---
n = 17 # Usando n pequeno para exemplo (2^17 - 1)
p = 2**n - 1

# 1. Geração de Chaves
F = low_hamming_weight_number(n, 3) # Secret Key
G = low_hamming_weight_number(n, 3)
R = secrets.randbits(n) # Random n-bit string
T = (F * R + G) % p    # Public Key: (R, T)

# 2. Mensagem (m)
m = int("10101010101010101", 2) 

# 3. Execução
C = mersenne_encrypt(m, n, 3, p, R, T)
m_final = mersenne_decrypt(C, F, p)

print(f"Mensagem Original:  {bin(m)}")
print(f"Mensagem Recuperada: {bin(m_final)}")

Mensagem Original:  0b10101010101010101
Mensagem Recuperada: 0b11001111001000110


## Com error correction, repetition code

In [10]:
import secrets

def low_hamming_weight_number(n, weight):
    bits = [0] * n
    indices = set()
    while len(indices) < weight:
        indices.add(secrets.randbelow(n))
    for i in indices:
        bits[i] = 1
    return int("".join(map(str, bits)), 2)

#codigo de reptição: subsitutui cada bit da mensagem por um bloco de bits (ex: 3 bits) para aumentar a redundância e permitir correção de erros
def repetition_encode(m_bits, k, n):
    """
    E(m): Repete cada bit da mensagem 'rep' vezes para preencher os n bits.
    """
    rep = n // k
    encoded = 0
    for i in range(k):
        # Isola o bit i da mensagem
        bit = (m_bits >> i) & 1
        if bit:
            # Cria um bloco de 'rep' bits 1 e coloca na posição i*rep
            mask = ((1 << rep) - 1) << (i * rep)
            encoded |= mask
    return encoded


def repetition_decode(c_star, k, n):
    """
    D(C): Divide os n bits em blocos e faz uma votação majoritária.
    """
    rep = n // k
    decoded = 0
    for i in range(k):
        # Extrai o bloco de bits correspondente ao bit original i
        block = (c_star >> (i * rep)) & ((1 << rep) - 1)
        # Conta quantos bits 1 existem (Hamming Weight do bloco)
        count_ones = bin(block).count('1')
        # Se mais de metade forem 1, o bit original era 1
        if count_ones > (rep // 2):
            decoded |= (1 << i)
    return decoded


def mersenne_encrypt(m_original, k, n, weight, p, R, T):
    # 1. Aplicar o Código de Repetição (E)
    m_encoded = repetition_encode(m_original, k, n)
    
    A = low_hamming_weight_number(n, weight)
    B1 = low_hamming_weight_number(n, weight)
    B2 = low_hamming_weight_number(n, weight)
    
    C1 = (A * R + B1) % p
    
    # 2. XOR com a mensagem expandida
    pad = (A * T + B2) % p
    C2 = pad ^ m_encoded 
    
    return (C1, C2)

def mersenne_decrypt(C, F, k, n, p):
    C1, C2 = C
    C2_star = (C1 * F) % p
    
    # 1. XOR para obter a mensagem com ruído
    diff = C2 ^ C2_star
    
    # 2. Aplicar a Decodificação por votação (D)
    m_recuperada = repetition_decode(diff, k, n)
    return m_recuperada

# --- Exemplo de Uso ---
n = 127 # n maior para permitir mais repetições e melhor correção
p = 2**n - 1
k = 7   # Vamos enviar apenas 7 bits de informação real
weight = 4 # Peso de Hamming das chaves/ruído

# 1. Geração de Chaves
F = low_hamming_weight_number(n, weight)
G = low_hamming_weight_number(n, weight)
R = secrets.randbits(n)
T = (F * R + G) % p

# 2. Mensagem (m) de k bits (ex: 0b1010101)
m_original = 0b1010101 

# 3. Execução
C = mersenne_encrypt(m_original, k, n, weight, p, R, T)
m_final = mersenne_decrypt(C, F, k, n, p)

print(f"Mensagem Original (k={k}): {bin(m_original)}")
print(f"Mensagem Recuperada:      {bin(m_final)}")
print(f"Sucesso: {m_original == m_final}")

Mensagem Original (k=7): 0b1010101
Mensagem Recuperada:      0b1010101
Sucesso: True


## BIT BY BIT encryption

In [14]:
import secrets
import math

def low_hamming_weight_number(n, h):
    """Gera um número de n bits com peso de Hamming exatamente h."""
    indices = set()
    while len(indices) < h:
        indices.add(secrets.randbelow(n))
    num = 0
    for i in indices:
        num |= (1 << i)
    return num

def hamming_weight(x):
    """Calcula o Peso de Hamming (quantidade de bits '1')."""
    return bin(x).count('1')

def generate_keys(n, p):
    """
    Gera a Chave Pública (H) e a Chave Secreta (G).
    H = F * G^-1 (mod p)
    """
    # Calcula o limite de h baseado em n: 4h^2 < n <= 16h^2
    # h_min > sqrt(n/16) e h_max < sqrt(n/4)
    h_min = math.ceil(math.sqrt(n / 16))
    h_max = math.floor(math.sqrt(n / 4))
    
    # Escolha aleatória de h dentro do limite
    h = secrets.choice(range(h_min, h_max + 1))
    
    F = low_hamming_weight_number(n, h)
    G = low_hamming_weight_number(n, h)
    
    # H = F / G mod p
    try:
        inv_G = pow(G, -1, p)
        H = (F * inv_G) % p
    except ValueError:
        # Caso raro onde G não é inversível, tentamos novamente
        return generate_keys(n, p)
        
    return (H, G, h)

def encrypt(pk_H, b, n, p, h):
    """
    C = (-1)^b * (A * H + B) mod p
    """
    A = low_hamming_weight_number(n, h)
    B = low_hamming_weight_number(n, h)
    
    val = (A * pk_H + B) % p
    
    if b == 0:
        return val
    else:
        # (-1) * val no mundo modular é p - val
        return (p - val) % p

def decrypt(C, sk_G, n, p, h):
    """
    Decriptação baseada na distância de Hamming.
    """
    # d = Ham(C * G mod p)
    target = (C * sk_G) % p
    d = hamming_weight(target)
    
    threshold = 2 * (h**2)
    
    if d <= threshold:
        return 0
    elif d >= n - threshold:
        return 1
    else:
        return None # Caso "?" (Indeterminado)

# --- Exemplo de Execução ---

# 1. Parâmetros (p deve ser um primo de Mersenne)
n = 127 
p = 2**n - 1

# 2. Setup de Chaves
pk, sk, h_usado = generate_keys(n, p)

# 3. Teste com bit 0
bit_original = 1
criptograma = encrypt(pk, bit_original, n, p, h_usado)
bit_recuperado = decrypt(criptograma, sk, n, p, h_usado)

print(f"Parâmetros: n={n}, h={h_usado}")
print(f"Bit Original: {bit_original}")
print(f"Criptograma (C): {hex(criptograma)[:20]}...")
print(f"Bit Recuperado: {bit_recuperado}")

Parâmetros: n=127, h=5
Bit Original: 1
Criptograma (C): 0x437512d129e13110fb...
Bit Recuperado: 1


## Section 4: Semantically Secure Public-Key Cryptosystem com repetition code

In [25]:
import secrets
import math

def low_hamming_weight_number(n, h):
    """Gera uma string de n bits com peso de Hamming exatamente h."""
    indices = set()
    while len(indices) < h:
        indices.add(secrets.randbelow(n))
    num = 0
    for i in indices:
        num |= (1 << i)
    return num

def repetition_encode(m_bits, k, n):
    """E(m): Código de correção de erro simples (Repetição)."""
    rep = n // k
    encoded = 0
    for i in range(k):
        bit = (m_bits >> i) & 1
        if bit:
            mask = ((1 << rep) - 1) << (i * rep)
            encoded |= mask
    return encoded

def repetition_decode(c_star, k, n):
    """D(C*): Decodificação por votação majoritária."""
    rep = n // k
    decoded = 0
    for i in range(k):
        block = (c_star >> (i * rep)) & ((1 << rep) - 1)
        if bin(block).count('1') > (rep // 2):
            decoded |= (1 << i)
    return decoded

def generate_keys(lambda_param):
    """
    Key Generation baseada no parâmetro de segurança lambda.
    h = lambda
    16h^2 >= n > 10h^2
    """
    h = lambda_param
    # Escolhemos um n que seja um primo de Mersenne no intervalo [10h^2, 16h^2]
    # Para lambda=16, 10h^2 = 2560. Um n comum seria 3217 ou similar.
    # Para este exemplo didático, vamos calcular n baseado em h:
    n_min = 10 * (h**2)
    n_max = 16 * (h**2)
    
    # Nota: Na prática, n deve ser um primo de Mersenne. 
    # Aqui usaremos um valor aproximado para manter o exemplo funcional.
    n = n_min + (n_max - n_min) // 2 
    p = 2**n - 1
    
    F = low_hamming_weight_number(n, h)
    G = low_hamming_weight_number(n, h)
    R = secrets.randbits(n)
    
    # pk := (R, T) onde T = F * R + G
    T = (F * R + G) % p
    
    return (R, T), F, n, p, h

def encrypt(pk, m, n, p, h, k):
    """
    Enc(pk, m) := (C1, C2)
    C1 = A * R + B1
    C2 = (A * T + B2) XOR E(m)
    """
    R, T = pk
    A = low_hamming_weight_number(n, h)
    B1 = low_hamming_weight_number(n, h)
    B2 = low_hamming_weight_number(n, h)
    
    # E(m)
    encoded_m = repetition_encode(m, k, n)
    
    C1 = (A * R + B1) % p
    pad = (A * T + B2) % p
    C2 = pad ^ encoded_m
    
    return (C1, C2)

def decrypt(C, sk_F, n, p, h, k):
    """
    Dec(sk, C) := D((F * C1) XOR C2)
    """
    C1, C2 = C
    
    # Calcula o termo que deve cancelar o pad: F * C1
    # F * (A * R + B1) = A * F * R + F * B1
    f_c1 = (sk_F * C1) % p
    
    # Extrai a mensagem com ruído usando XOR
    noisy_m = f_c1 ^ C2
    
    # D(noisy_m)
    return repetition_decode(noisy_m, k, n)

# --- Execução ---
# Parâmetro de segurança (ex: lambda = 16)
lambda_sec = 16 
k_message_size = 16 # Tamanho do bloco m

# 1. Setup
pk, sk, n, p, h = generate_keys(lambda_sec)

# 2. Mensagem m (Ex: 16 bits aleatórios)
m_original = secrets.randbits(k_message_size)

# 3. Cifrar
C = encrypt(pk, m_original, n, p, h, k_message_size)

# 4. Decifrar
m_recuperada = decrypt(C, sk, n, p, h, k_message_size)

print(f"--- Sistema de Bloco (Mersenne) ---")
print(f"n: {n}, h: {h}, k: {k_message_size}")
print(f"Mensagem Original:  {bin(m_original)}")
print(f"Mensagem Recuperada: {bin(m_recuperada)}")
print(f"Sucesso: {m_original == m_recuperada}")

--- Sistema de Bloco (Mersenne) ---
n: 3328, h: 16, k: 16
Mensagem Original:  0b100011001010111
Mensagem Recuperada: 0b100011001010111
Sucesso: True


## Mersenne Key Encapsulation Mechanism